# T3.8 확인: Colab과 Kaggle에서 같은 명령으로 1에폭 실행

`scripts/run_experiment.py`를 **실제 CLI 그대로** Colab T4와 Kaggle에서 실행해, 로컬 CPU 테스트로는 확인할 수 없는 항목을 봅니다. 로직은 `src/`에 있고 이 노트북은 실행·확인만 합니다.

- 데이터: T1.5에서 올린 **파일럿용 압축 파일**(`div2k_x4_p96_n10000_s42_e8bc539e`)을 씁니다. 이 노트북 안에서만 쓰는 **점검용 공정 비교 조건 사본**(`pilot_determined`를 점검용 값으로 채움, 에폭 1)을 `/content`(Colab) 또는 `/kaggle/working`(Kaggle) 아래에 만듭니다. `configs/`는 읽기만 하며 이 값은 공정 비교 조건이 아닙니다(`pilot_determined` 확정은 T4.2).
- 결과(`--output-dir`)와 체크포인트는 임시 폴더에만 쓰고 `results/`에는 아무것도 쓰지 않습니다.

## 어디서 무엇을 실행하나
| 환경 | 실행할 셀 | 확인하는 것 |
|---|---|---|
| **Colab T4** | 1 → 2 → 3 → 4(a) → 5(b) → 6 | (a) Drive식 `--data-archive` 해제 + 1에폭 학습·평가(CUDA, 실제 AlexNet LPIPS)와 맞지 않는 재개 요청을 압축 해제 전에 거부하는지, (b) 같은 데이터를 사전 해제 폴더로 줘서(Kaggle 방식) 같은 결과인지 |
| **Kaggle** (GPU 가속기, 인터넷 켜기, 비공개 Dataset을 Add Input) | 1 → 3 → 5(b) → 6 | `--data-dir /kaggle/input`만으로 같은 CLI가 1에폭을 끝까지 도는지 |

Colab 준비: 런타임 > 런타임 유형 변경 > **T4 GPU**. 데이터 압축 파일은 내 드라이브나 공유 드라이브에서 이름으로 찾습니다. 다른 사람이 "공유 문서함"으로 준 파일이면 먼저 "내 드라이브에 바로가기 추가"를 하거나 셀 2의 `ARCHIVE_PATH`에 Drive 안 경로를 적습니다.

6번 셀의 출력을 **그대로 복사해 전달**합니다(Colab·Kaggle 각각). 통과 조건은 두 환경 모두 `전체: PASS`입니다.

주의: 저장소가 Public이라 공유 링크·Dataset 주소는 출력하지 않습니다. 실행 후 이 노트북의 셀 출력은 지우고 커밋합니다.

In [ ]:
# 1. 환경 감지, 저장소 가져오기
import json, os, platform, shutil, subprocess, sys, time, zipfile
from pathlib import Path

REPO_URL = "https://github.com/minyi-k03/VideoProcessing-Project.git"
BRANCH = "YongMin"
IN_KAGGLE = Path("/kaggle/input").exists()
WORK = Path("/kaggle/working") if IN_KAGGLE else Path("/content")
REPO = WORK / "repo"
ROOT = WORK / "t3_8_check"
RESULTS = {}

if (REPO / ".git").is_dir():   # 다시 실행하면 BRANCH의 최신 커밋으로 맞춘다(이 노트북 전용 clone이라 로컬 변경 없음)
    subprocess.run(["git", "-C", str(REPO), "fetch", "-q", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "-q", "-B", BRANCH, "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "log", "--oneline", "-3"], check=True)   # T3.8 커밋이 보여야 한다. 안 보이면 로컬에서 git push 먼저
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements.txt")], check=True)
assert (REPO / "scripts" / "run_experiment.py").is_file() and (REPO / "src" / "sr_pipeline" / "engine" / "runtime_data.py").is_file(), \
    "T3.8 코드가 없습니다: 로컬에서 git push 한 뒤 다시 실행하세요."
sys.path.insert(0, str(REPO / "src"))
for name in [m for m in sys.modules if m == "sr_pipeline" or m.startswith("sr_pipeline.")]:
    del sys.modules[name]   # 이전 실행에서 import한 옛 모듈이 남지 않게

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
COMMIT = subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"], capture_output=True, text=True, check=True).stdout.strip()
print("환경:", "Kaggle" if IN_KAGGLE else "Colab", "| device:", DEVICE, "| torch", torch.__version__, "|",
      torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU")
print("commit", COMMIT)
if not IN_KAGGLE:
    assert DEVICE == "cuda", "GPU 런타임이 아님: 런타임 > 런타임 유형 변경 > T4 GPU"

In [ ]:
# 2. (Colab만) 공유 Drive의 압축 파일을 런타임 로컬로 복사 — 노트북 02 셀 3과 같은 동작
assert not IN_KAGGLE, "Kaggle에서는 이 셀을 건너뜁니다(셀 3으로)."
import glob
from google.colab import drive
drive.mount("/content/drive")
ARCHIVE_NAME = "div2k_x4_p96_n10000_s42_e8bc539e.zip"
ARCHIVE_PATH = ""   # 비워 두면 내 드라이브·공유 드라이브에서 이름으로 찾는다. 못 찾으면 "/content/drive/..." 경로를 직접 적는다.
if ARCHIVE_PATH:
    matches = [ARCHIVE_PATH] if Path(ARCHIVE_PATH).is_file() else []
else:
    matches = [path for base in ("/content/drive/MyDrive", "/content/drive/Shareddrives")
               for path in glob.glob(f"{base}/**/{ARCHIVE_NAME}", recursive=True)]
assert len(matches) == 1, f"Drive에서 {ARCHIVE_NAME}을 {len(matches)}개 찾았습니다(1개여야 함). ARCHIVE_PATH로 지정하세요."
shutil.rmtree(ROOT, ignore_errors=True)       # 이 노트북의 산출물만 지운다
(ROOT / "drive_copy").mkdir(parents=True)
LOCAL_ARCHIVE = ROOT / "drive_copy" / "sr_data.zip"
shutil.copyfile(matches[0], LOCAL_ARCHIVE)
print("복사 완료:", f"{LOCAL_ARCHIVE.stat().st_size / 1e6:.0f} MB")

In [ ]:
# 3. 점검용 공정 비교 조건 사본 만들기. 패치 풀 메타데이터(id·크기)는 실제 압축 파일/Dataset에서 읽고, 학습 설정은 점검용 값이다.
import yaml
from sr_pipeline.engine.runtime_data import find_data_root

if IN_KAGGLE:
    shutil.rmtree(ROOT, ignore_errors=True)
    ROOT.mkdir(parents=True)
    KAGGLE_DATA_DIR = Path("/kaggle/input")                      # 노트북 02와 같은 인자
    data_root = find_data_root(KAGGLE_DATA_DIR)
    metadata = json.loads((data_root / "patch_pool" / "metadata.json").read_text(encoding="utf-8"))
    testsets_dir = data_root / "testsets"
else:
    with zipfile.ZipFile(LOCAL_ARCHIVE) as archive:
        metadata = json.loads(archive.read("patch_pool/metadata.json").decode("utf-8"))
        member_names = archive.namelist()

FAIR = yaml.safe_load((REPO / "configs" / "fair_comparison.yaml").read_text(encoding="utf-8"))

def hr_image_ids(dataset):
    if IN_KAGGLE:
        return sorted(p.stem for p in (testsets_dir / dataset / "HR").iterdir())
    return sorted(Path(n).stem for n in member_names if n.startswith(f"testsets/{dataset}/HR/") and not n.endswith("/"))

representative = {}
for dataset in FAIR["fixed_rules"]["datasets"]:
    ids = hr_image_ids(dataset)
    representative[dataset] = {"median": ids[len(ids) // 2], "worst": ids[0]}   # 점검용 선택(규칙으로 고른 대표 영상 아님)
FAIR["pilot_determined"] = {
    "patch_pool_id": metadata["patch_pool_id"], "patch_pool_size": metadata["num_patches"], "patch_size_hr": metadata["patch_size_hr"],
    "patches_per_epoch": 256, "epochs": 1, "batch_size": 16, "learning_rate": 1e-3,      # 점검용 값(공정 비교 조건 아님)
    "optimizer": "adam", "scheduler": "none", "augmentation": "rot90x4_hflip",
    "edge_loss_lambda": 0.1, "edge_top_q": 10, "representative_images": representative,
}
FAIR_PATH = ROOT / "fair_for_check.yaml"
FAIR_PATH.write_text(yaml.safe_dump(FAIR, allow_unicode=True), encoding="utf-8")
print("pool:", metadata["patch_pool_id"], "| 패치", metadata["num_patches"], "| 점검용 사본 작성 완료")


def run_cli(label, *extra, config="baseline", seed=0):
    """실제 CLI를 부른다. 경로 인자(--data-dir/--data-archive)만 환경마다 다르다."""
    out_dir, ckpt_dir = ROOT / label / "records", ROOT / label / "checkpoints"
    shutil.rmtree(ROOT / label, ignore_errors=True)
    cmd = [sys.executable, "scripts/run_experiment.py", "--config", config, "--seed", str(seed),
           "--fair-config", str(FAIR_PATH), "--output-dir", str(out_dir), "--checkpoint-dir", str(ckpt_dir),
           "--device", DEVICE, *extra]
    started = time.time()
    proc = subprocess.run(cmd, cwd=REPO, env={**os.environ, "PYTHONPATH": str(REPO / "src")}, text=True, capture_output=True)
    print(proc.stdout)
    if proc.returncode != 0:
        print(proc.stderr[-3000:])
    print(f"소요 {time.time() - started:.0f}초")
    return proc, out_dir, ckpt_dir


def check_record(label, proc, out_dir, ckpt_dir):
    """실행 기록의 형식과 내용을 확인하고 모두 맞으면 True."""
    import pandas as pd
    checks = {"CLI 종료 코드 0": proc.returncode == 0}
    if proc.returncode == 0:
        (run_dir,) = [p for p in out_dir.iterdir() if p.is_dir()]
        summary, per_image = pd.read_csv(run_dir / "summary.csv"), pd.read_csv(run_dir / "per_image.csv")
        checks["summary 4행(테스트셋 4종), status complete"] = (
            list(summary["dataset"]) == FAIR["fixed_rules"]["datasets"] and set(summary["status"]) == {"complete"})
        checks["commit이 실행한 코드 커밋과 같음(-dirty 아님)"] = set(summary["commit"]) == {COMMIT}
        checks["per_image 행 존재, 지표 유한"] = len(per_image) > 0 and bool(
            pd.notna(per_image[["psnr_y", "ssim_y", "psnr_rgb", "edge_psnr", "lpips"]]).all().all())
        checks["train_log.json, weights.pt 저장"] = (
            (run_dir / "train_log.json").is_file() and (ckpt_dir / run_dir.name / "weights.pt").is_file())
        checks["출력과 기록에 저장 경로 없음"] = str(ROOT) not in proc.stdout + (run_dir / "train_log.json").read_text(encoding="utf-8")
        RESULTS[label] = {"summary": summary}
    for name, value in checks.items():
        print("OK  " if value else "FAIL", name)
    return all(checks.values())

In [ ]:
# 4(a). (Colab) Drive식: 복사한 zip을 --data-archive로 풀어 1에폭 학습·평가.
# CUDA에서 학습 직후 평가까지(결정적 알고리즘 strict 아래, 실제 AlexNet LPIPS) 오류 없이 끝나야 한다.
assert not IN_KAGGLE
proc, out_dir, ckpt_dir = run_cli("a_drive", "--data-dir", str(ROOT / "a_data"), "--data-archive", str(LOCAL_ARCHIVE))
RESULTS["(a) Drive식 --data-archive"] = check_record("a_drive", proc, out_dir, ckpt_dir)

# 이어 갈 수 없는 --resume-run-id는 큰 압축 파일을 풀기 전에 거부해야 한다(실패가 정상이라 아래 오류 출력은 예상된 것).
probe_data = ROOT / "resume_probe_data"
shutil.rmtree(probe_data, ignore_errors=True)
proc, _, _ = run_cli("resume_probe", "--data-dir", str(probe_data), "--data-archive", str(LOCAL_ARCHIVE),
                     "--resume-run-id", "not-this-run-baseline-s0")
RESULTS["(a) 맞지 않는 재개 요청은 압축 해제 전에 거부"] = (
    proc.returncode != 0 and "resume_run_id" in proc.stderr and not probe_data.exists())
print("OK  " if RESULTS["(a) 맞지 않는 재개 요청은 압축 해제 전에 거부"] else "FAIL", "재개 요청 사전 거부")

In [ ]:
# 5(b). Kaggle식: 사전 해제된 폴더를 --data-dir로만 준다(--data-archive 없음). 노트북 02의 Kaggle 경로와 같은 인자다.
#   Kaggle: /kaggle/input 그대로.   Colab: (a)가 푼 폴더를 입력 폴더처럼 쓴다.
data_dir_b = str(KAGGLE_DATA_DIR) if IN_KAGGLE else str(ROOT / "a_data")
proc, out_dir, ckpt_dir = run_cli("b_prepared", "--data-dir", data_dir_b)
RESULTS["(b) Kaggle식 --data-dir"] = check_record("b_prepared", proc, out_dir, ckpt_dir)

In [ ]:
# 6. 요약 (이 출력을 그대로 복사해 전달: T3.8 통과 근거, Colab·Kaggle 각각)
timing = ["train_minutes", "infer_ms_net", "infer_ms_total"]
if "a_drive" in RESULTS and "b_prepared" in RESULTS:   # Colab: 같은 데이터·구성·시드면 입력 방식과 무관하게 같은 결과(GPU 결정성은 T3.7에서 확인)
    RESULTS["(a)와 (b)의 summary가 같음(시간 필드 제외)"] = bool(
        RESULTS["a_drive"]["summary"].drop(columns=timing).equals(RESULTS["b_prepared"]["summary"].drop(columns=timing)))
print("platform :", platform.platform())
print("environ  :", "Kaggle" if IN_KAGGLE else "Colab", "| device", DEVICE, "| torch", torch.__version__)
print("commit   :", COMMIT)
verdicts = {name: value for name, value in RESULTS.items() if isinstance(value, bool)}
for name, value in verdicts.items():
    print("PASS" if value else "FAIL", name)
for key in ("a_drive", "b_prepared"):
    if key in RESULTS:
        display(RESULTS[key]["summary"][["dataset", "psnr_y", "ssim_y", "lpips", "params", "train_minutes", "status"]])
print("전체:", "PASS" if verdicts and all(verdicts.values()) else "FAIL")